# AUPRC for 80/20 Models

This notebook calculates the area under the precision-recall curve (AUPRC) for the saved 80/20 models using the prediction CSVs already created during evaluation.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve

MODEL_DIR = Path('../output/models/hypertuning')
FIGURE_DIR = Path('../figures/pretrained_8020')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = ['ET', 'RF', 'XGB', 'LGBM', 'LogReg', 'KNN']
model_paths = {name: MODEL_DIR / f'{name}_8020_best_model.pkl' for name in MODEL_NAMES}
model_paths

## Load the 80/20 Models

The AUPRC values below are calculated from the saved prediction CSV. Loading the models here is just a quick check that the expected 80/20 model files are present and readable.

In [ ]:
loaded_models = {}

for name, path in model_paths.items():
    if not path.exists():
        print(f'{name}: missing ({path})')
        continue

    try:
        loaded_models[name] = joblib.load(path)
        print(f'{name}: loaded')
    except Exception as exc:
        print(f'{name}: could not load ({exc})')

list(loaded_models)

## Load Existing 80/20 Predictions

AUPRC uses the true label and each model's prediction score/probability. The combined prediction file should contain `model`, `y_true`, and `y_score` columns.

In [ ]:
predictions_path = MODEL_DIR / 'predictions_all_models_existing_8020.csv'

predictions = pd.read_csv(predictions_path)
predictions.head()

In [ ]:
required_columns = {'model', 'y_true', 'y_score'}
missing_columns = required_columns - set(predictions.columns)

if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

auprc_rows = []

for model_name, group in predictions.groupby('model'):
    y_true = group['y_true']
    y_score = group['y_score']
    auprc = average_precision_score(y_true, y_score)
    auprc_rows.append({'model': model_name, 'auprc': auprc})

auprc_df = pd.DataFrame(auprc_rows).sort_values('auprc', ascending=False)
auprc_df

## Combined Precision-Recall Curves

In [ ]:
plt.figure(figsize=(8, 6))

for model_name, group in predictions.groupby('model'):
    y_true = group['y_true']
    y_score = group['y_score']
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    plt.plot(recall, precision, linewidth=2, label=f'{model_name} (AUPRC={auprc:.3f})')

baseline = predictions['y_true'].mean()
plt.axhline(baseline, color='gray', linestyle='--', linewidth=1.5, label=f'Baseline={baseline:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves for 80/20 Models')
plt.legend(loc='lower left', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()

figure_path = FIGURE_DIR / 'auprc_curves_combined.png'
plt.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

figure_path

## Add AUPRC to the Existing Metrics CSV

In [ ]:
metrics_path = MODEL_DIR / 'results_existing_8020_official.csv'
metrics = pd.read_csv(metrics_path)

metrics_with_auprc = metrics.drop(columns=['auprc'], errors='ignore').merge(
    auprc_df,
    on='model',
    how='left',
)

output_path = MODEL_DIR / 'results_existing_8020_official_with_auprc.csv'
metrics_with_auprc.to_csv(output_path, index=False)

metrics_with_auprc

## Three-Trial AUPRC Summary

This recomputes AUPRC for each model in each trial, then summarizes the mean and standard deviation across the three trials. In sklearn, `average_precision_score` is the standard step-wise estimate used for AUPRC.

In [ ]:
trial_predictions_path = MODEL_DIR / 'trial_predictions_3runs_existing_8020.csv'
trial_predictions = pd.read_csv(trial_predictions_path)

required_trial_columns = {'trial', 'seed', 'model', 'y_true', 'y_score'}
missing_trial_columns = required_trial_columns - set(trial_predictions.columns)

if missing_trial_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_trial_columns)}')

trial_auprc_rows = []

for (trial, seed, model_name), group in trial_predictions.groupby(['trial', 'seed', 'model']):
    auprc = average_precision_score(group['y_true'], group['y_score'])
    baseline = group['y_true'].mean()
    trial_auprc_rows.append({
        'trial': trial,
        'seed': seed,
        'model': model_name,
        'auprc': auprc,
        'baseline_precision': baseline,
    })

trial_auprc_df = pd.DataFrame(trial_auprc_rows).sort_values(['model', 'trial'])
trial_auprc_df

In [ ]:
auprc_summary = (
    trial_auprc_df
    .groupby('model', as_index=False)
    .agg(
        auprc_mean=('auprc', 'mean'),
        auprc_std=('auprc', 'std'),
        baseline_precision_mean=('baseline_precision', 'mean'),
        baseline_precision_std=('baseline_precision', 'std'),
    )
    .sort_values('auprc_mean', ascending=False)
)

trial_auprc_output_path = MODEL_DIR / 'trial_auprc_3runs_existing_8020.csv'
trial_auprc_summary_path = MODEL_DIR / 'trials_auprc_summary_mean_std_existing_8020.csv'

trial_auprc_df.to_csv(trial_auprc_output_path, index=False)
auprc_summary.to_csv(trial_auprc_summary_path, index=False)

auprc_summary

## Add AUPRC Columns to the Full Trial Summary Table

In [ ]:
trial_summary_path = MODEL_DIR / 'trials_summary_mean_std_existing_8020.csv'
trial_summary = pd.read_csv(trial_summary_path)

trial_summary_with_auprc = trial_summary.drop(
    columns=['auprc_mean', 'auprc_std', 'baseline_precision_mean', 'baseline_precision_std'],
    errors='ignore',
).merge(
    auprc_summary,
    on='model',
    how='left',
)

trial_summary_with_auprc_path = MODEL_DIR / 'trials_summary_mean_std_existing_8020_with_auprc.csv'
trial_summary_with_auprc.to_csv(trial_summary_with_auprc_path, index=False)

trial_summary_with_auprc

In [ ]:
print(f'Saved combined AUPRC plot to: {figure_path}')
print(f'Saved metrics with AUPRC to: {output_path}')
print(f'Saved per-trial AUPRC values to: {trial_auprc_output_path}')
print(f'Saved trial AUPRC mean/std summary to: {trial_auprc_summary_path}')
print(f'Saved full trial summary with AUPRC columns to: {trial_summary_with_auprc_path}')